# Parking Sign Detection - Interactive Inference (Colab)

Load pretrained model, upload images, and interactively adjust predictions.

**Features:**
- Batch image upload
- Confidence filtering
- Interactive box adjustment
- Export annotations

## 1. Mount Google Drive (Optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Setup

In [ ]:
!pip install ultralytics -q

from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import numpy as np
from PIL import Image
from google.colab import files
import shutil

# Paths
UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Upload directory: {UPLOAD_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 3. Upload Model Weights

Upload your trained `best.pt` file.

In [ ]:
print("Upload your trained model weights (best.pt):")
uploaded = files.upload()

if uploaded:
    MODEL_PATH = list(uploaded.keys())[0]
    print(f"Model uploaded: {MODEL_PATH}")
else:
    print("No file uploaded. Using default yolo11n.pt for testing.")
    MODEL_PATH = "yolo11n.pt"

## 4. Load Model

In [ ]:
model = YOLO(MODEL_PATH)
print(f"Model loaded: {MODEL_PATH}")
print(f"Classes: {model.names}")

## 5. Upload Test Images

In [ ]:
print("Upload test images (you can select multiple):")
uploaded_imgs = files.upload()

uploaded_files = []
for filename in uploaded_imgs.keys():
    img_path = UPLOAD_DIR / filename
    shutil.copy(filename, img_path)
    uploaded_files.append(str(img_path))
    print(f"Saved: {filename}")

print(f"\nTotal images uploaded: {len(uploaded_files)}")

## 6. Run Predictions

In [ ]:
# Confidence threshold slider
conf_threshold = widgets.FloatSlider(
    value=0.25,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Confidence:',
    continuous_update=False
)

predict_button = widgets.Button(description='Run Predictions', button_style='primary')
predict_output = widgets.Output()

# Storage for predictions
predictions = {}  # {image_path: {boxes: [], confidences: [], classes: []}}

def run_predictions(b):
    global predictions
    predictions = {}
    conf = conf_threshold.value
    
    with predict_output:
        clear_output()
        if not uploaded_files:
            print("Please upload images first.")
            return
        
        print(f"Running predictions with confidence threshold: {conf}")
        print("-" * 50)
        
        for img_path in uploaded_files:
            results = model.predict(img_path, conf=conf, verbose=False)
            
            boxes = []
            confidences = []
            classes = []
            
            for r in results:
                if r.boxes is not None:
                    for box in r.boxes:
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                        boxes.append([float(x1), float(y1), float(x2), float(y2)])
                        confidences.append(float(box.conf.cpu().numpy()))
                        classes.append(int(box.cls.cpu().numpy()))
            
            predictions[img_path] = {
                'boxes': boxes,
                'confidences': confidences,
                'classes': classes
            }
            
            print(f"{Path(img_path).name}: {len(boxes)} detections")
        
        print("\nPredictions complete!")

predict_button.on_click(run_predictions)
display(widgets.VBox([conf_threshold, predict_button, predict_output]))

## 7. Interactive Visualization

In [ ]:
# Navigation and adjustment UI
current_idx = 0
adjusted_boxes = {}  # Store manually adjusted boxes

# Navigation widgets
prev_button = widgets.Button(description='◀ Prev', button_style='info')
next_button = widgets.Button(description='Next ▶', button_style='info')
img_label = widgets.Label(value='')

# Display widget
viz_output = widgets.Output()

def show_image(idx=None):
    global current_idx
    if idx is not None:
        current_idx = idx
    
    if not uploaded_files:
        with viz_output:
            clear_output()
            print("No images loaded.")
        return
    
    current_idx = max(0, min(current_idx, len(uploaded_files) - 1))
    img_path = uploaded_files[current_idx]
    
    img_label.value = f"Image {current_idx + 1}/{len(uploaded_files)}: {Path(img_path).name}"
    
    img = Image.open(img_path)
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img)
    
    if img_path in predictions:
        boxes = predictions[img_path]['boxes']
        confidences = predictions[img_path]['confidences']
        classes = predictions[img_path]['classes']
        
        if img_path in adjusted_boxes:
            boxes = adjusted_boxes[img_path]
        
        colors = ['red', 'blue', 'green', 'yellow', 'purple', 'cyan', 'orange', 'pink']
        
        for i, (box, conf, cls) in enumerate(zip(boxes, confidences, classes)):
            x1, y1, x2, y2 = box
            color = colors[i % len(colors)]
            
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            
            label = f"{model.names.get(cls, cls)}: {conf:.2f}"
            ax.text(x1, y1 - 5, label, color=color, 
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    
    ax.set_title(Path(img_path).name)
    ax.axis('off')
    plt.tight_layout()
    
    with viz_output:
        clear_output(wait=True)
        plt.show()

def go_next(b):
    global current_idx
    current_idx += 1
    show_image()

def go_prev(b):
    global current_idx
    current_idx -= 1
    show_image()

prev_button.on_click(go_prev)
next_button.on_click(go_next)

display(widgets.HBox([prev_button, img_label, next_button]))
display(viz_output)

if uploaded_files:
    show_image(0)

### 7.1 Manual Box Adjustment

Enter box coordinates to manually adjust detections. Format: `x1,y1,x2,y2`

In [ ]:
# Box adjustment widget
box_input = widgets.Text(
    placeholder='x1,y1,x2,y2 (e.g., 100,100,300,400)',
    description='Box coords:',
    style={'description_width': 'initial'}
)

box_index = widgets.IntSlider(
    value=0,
    min=0,
    max=10,
    description='Box index:',
    continuous_update=False
)

adjust_button = widgets.Button(description='Update Box', button_style='warning')
adjust_output = widgets.Output()

def update_box(b):
    if not uploaded_files:
        return
    
    img_path = uploaded_files[current_idx]
    idx = box_index.value
    
    with adjust_output:
        clear_output()
        
        if img_path not in predictions:
            print("No predictions for this image.")
            return
        
        boxes = predictions[img_path]['boxes'].copy()
        
        if idx >= len(boxes):
            print(f"Box index {idx} out of range (0-{len(boxes)-1})")
            return
        
        try:
            coords = [float(x.strip()) for x in box_input.value.split(',')]
            if len(coords) != 4:
                raise ValueError("Need 4 coordinates")
            
            if img_path not in adjusted_boxes:
                adjusted_boxes[img_path] = boxes.copy()
            
            adjusted_boxes[img_path][idx] = coords
            print(f"Updated box {idx}: {coords}")
            show_image()
        except Exception as e:
            print(f"Error: {e}")

adjust_button.on_click(update_box)
display(widgets.VBox([box_index, box_input, adjust_button, adjust_output]))

## 8. View Predictions Data

In [ ]:
import pandas as pd

rows = []
for img_path, pred in predictions.items():
    for box, conf, cls in zip(pred['boxes'], pred['confidences'], pred['classes']):
        rows.append({
            'Image': Path(img_path).name,
            'Class': model.names.get(cls, cls),
            'Confidence': f"{conf:.3f}",
            'Box': f"[{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]"
        })

df = pd.DataFrame(rows)
display(df)

## 9. Export Results

In [ ]:
export_button = widgets.Button(description='Export Annotations', button_style='success')
export_output = widgets.Output()

def export_results(b):
    with export_output:
        clear_output()
        
        if not predictions:
            print("No predictions to export.")
            return
        
        # Export JSON
        export_data = []
        for img_path, pred in predictions.items():
            boxes = adjusted_boxes.get(img_path, pred['boxes'])
            export_data.append({
                'image': Path(img_path).name,
                'boxes': boxes,
                'confidences': pred['confidences'],
                'classes': pred['classes']
            })
        
        json_path = OUTPUT_DIR / 'predictions.json'
        with open(json_path, 'w') as f:
            json.dump(export_data, f, indent=2)
        print(f"Exported: {json_path}")
        
        # Save annotated images
        for img_path in uploaded_files:
            img = Image.open(img_path)
            fig, ax = plt.subplots(1, 1, figsize=(12, 10))
            ax.imshow(img)
            
            if img_path in predictions:
                boxes = adjusted_boxes.get(img_path, predictions[img_path]['boxes'])
                confidences = predictions[img_path]['confidences']
                classes = predictions[img_path]['classes']
                
                for box, conf, cls in zip(boxes, confidences, classes):
                    x1, y1, x2, y2 = box
                    rect = patches.Rectangle(
                        (x1, y1), x2 - x1, y2 - y1,
                        linewidth=3, edgecolor='red', facecolor='none'
                    )
                    ax.add_patch(rect)
                    label = f"{model.names.get(cls, cls)}: {conf:.2f}"
                    ax.text(x1, y1 - 5, label, color='red', fontsize=12,
                           bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))
            
            ax.axis('off')
            plt.tight_layout()
            
            output_img_path = OUTPUT_DIR / f"annotated_{Path(img_path).name}"
            plt.savefig(output_img_path, bbox_inches='tight', dpi=150)
            plt.close()
            print(f"Saved: {output_img_path}")
        
        print(f"\nExport complete! Files in {OUTPUT_DIR}")

export_button.on_click(export_results)
display(widgets.VBox([export_button, export_output]))

## 10. Download Results

In [ ]:
print("Downloading output files...")
!zip -r /content/output.zip /content/output/*
files.download('/content/output.zip')

## Summary

**Usage:**
1. (Optional) Mount Google Drive
2. Upload model weights (`best.pt`)
3. Upload test images
4. Adjust confidence threshold
5. Run predictions
6. Navigate through results (Next/Prev)
7. Manually adjust boxes if needed
8. Export annotated images + JSON
9. Download results as ZIP

**Note:** To change confidence threshold, adjust slider and re-run predictions.